## 1. Configuration — change paths here

In [1]:
import os

# ---------------- PATHS (edit these if you move the notebook / dataset) ----------------
DATASET_ROOT  = "/home/haseebumer/content/data/UCF_Crimes_MINI"   # Videos/ + Anomaly_Detection_splits/
SPLITS_DIR    = os.path.join(DATASET_ROOT, "Anomaly_Detection_splits")
RUNS_DIR      = "./runs_ucf_fullvideo"                           # separate from ./runs_ucf_mini (clip-sampling model)
RESUME        = True                                             # True: continue from runs_ucf_fullvideo/*_last.pt if it exists

# ---------------- SPLITS ----------------
VAL_FRACTION = 0.15        # part of Anomaly_Train.txt held out for model selection (stratified by category)

# ---------------- WHOLE-VIDEO INPUT ----------------
# EVERY frame of a video goes into the model (no frame skipping, no random/sampled clips).
# R(2+1)D-18 works on 16-frame inputs, so a video is cut into consecutive, non-overlapping 16-frame clips that
# together cover all of its frames; the features of all clips are averaged into one video feature -> one prediction.
CLIP_LEN     = 16          # frames per clip (what R(2+1)D-18 was pretrained with)
RESIZE_HW    = (128, 171)  # resize (H, W) before cropping, as in torchvision's Kinetics recipe
CROP         = 112
CLIPS_PER_SEGMENT = 32     # a video is decoded and sent to the GPU 32 clips (512 frames) at a time
FT_CHUNK     = 16          # stage 2: clips per layer4 forward/backward (bounds VRAM)

# ---------------- TRAINING ----------------
# One video = one sample. Videos have very different lengths (200 to 600k frames), so they are not stacked into
# batches; instead the gradients of ACCUM_VIDEOS videos are accumulated before each optimizer step.
ACCUM_VIDEOS = 8
NUM_WORKERS  = 8           # CPU decode processes, 1 FFmpeg thread each
PREFETCH     = 2           # segments queued per worker -> caps the host RAM held by the loader

HEAD_EPOCHS = 15
FT_EPOCHS   = 10
LR_HEAD     = 1e-3         # stage 1 head LR
LR_FT_HEAD  = 3e-4         # stage 2 head LR
LR_FT_L4    = 1e-4         # stage 2 layer4 LR
WEIGHT_DECAY = 1e-4

SEED = 42

os.makedirs(RUNS_DIR, exist_ok=True)
assert os.path.isdir(DATASET_ROOT), f"DATASET_ROOT not found: {DATASET_ROOT}"
assert os.path.isdir(SPLITS_DIR), f"SPLITS_DIR not found: {SPLITS_DIR}"

## 2. Imports and GPU setup

In [2]:
import random, glob, time, copy
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True                 # few input shapes (1..32 clips) -> let cuDNN pick fastest conv3d kernels
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
AMP_DTYPE = torch.bfloat16                            # RTX 50-series supports bf16 natively (no GradScaler needed)

print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(torch.cuda.get_device_name(0), f"| free {free/2**30:.1f} GB / {total/2**30:.1f} GB")


def mem_status():
    """GPU memory held by this process (shows up in nvidia-smi) and host RAM still available."""
    ram = int(open("/proc/meminfo").read().split("MemAvailable:")[1].split()[0]) / 2**20
    return f"VRAM peak {torch.cuda.max_memory_reserved() / 2**30:.1f} GB | RAM free {ram:.1f} GB"

/home/haseebumer/anaconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cu130 | device: cuda
NVIDIA GeForce RTX 5080 | free 14.3 GB / 15.6 GB


## 3. Build the file lists (0 = NORMAL, 1 = ANOMALY; VAL is a stratified hold-out from the train list; TEST has no labels)

In [3]:
CLASSES = ["NORMAL", "ANOMALY"]

def category(path):
    """Folder name of the video, e.g. 'Robbery' or 'Training_Normal_Videos_Anomaly'."""
    return os.path.basename(os.path.dirname(path))

def read_split(fname, labelled=True):
    """labelled=False: label is -1 and is never used (TEST is treated like unseen real-world data)."""
    items = []
    for line in open(os.path.join(SPLITS_DIR, fname)):
        rel = line.strip()
        if not rel:
            continue
        path = os.path.join(DATASET_ROOT, "Videos", rel)
        assert os.path.isfile(path), f"missing video: {path}"
        items.append((path, (0 if "Normal" in category(path) else 1) if labelled else -1))
    return items


train_all = read_split("Anomaly_Train.txt")
test_items = read_split("Anomaly_Test.txt", labelled=False)
train_items, val_items = train_test_split(train_all, test_size=VAL_FRACTION, random_state=SEED,
                                          stratify=[category(p) for p, _ in train_all])
train_items, val_items = sorted(train_items), sorted(val_items)

for name, it in [("TRAIN", train_items), ("VAL", val_items)]:
    labs = [l for _, l in it]
    print(f"{name:5s}: {len(it):5d} videos | anomaly {sum(labs):4d} | normal {len(labs)-sum(labs):4d}")
print(f"TEST : {len(test_items):5d} videos | unlabelled (predictions only)")
print(pd.Series([category(p) for p, _ in train_all]).value_counts().to_string())

TRAIN:   297 videos | anomaly  165 | normal  132
VAL  :    53 videos | anomaly   29 | normal   24
TEST :    84 videos | unlabelled (predictions only)
Training_Normal_Videos_Anomaly    156
Robbery                            40
RoadAccidents                      26
Burglary                           21
Stealing                           19
Assault                            16
Abuse                              13
Explosion                          12
Arrest                             10
Fighting                           10
Vandalism                          10
Shoplifting                         7
Shooting                            6
Arson                               4


## 4. Whole-video dataset

- **The whole video goes into the model.** Every frame is used (no frame stride, no sampled clips): the video is cut into consecutive, non-overlapping 16-frame clips that cover all of its frames (the last clip is padded by repeating the last frame). The model's 512-d features of all these clips are averaged into one video feature, and the classifier makes **one prediction per video** from it.
- A video can be very long (the longest has ~628k frames ≈ 5.8 h), far too big for RAM or VRAM in one piece, so it is **streamed**: workers decode it in segments of `CLIPS_PER_SEGMENT` clips, the GPU adds up the clip features segment by segment, and the video's prediction is made when its last segment arrives. A custom sampler sends the segments of one video consecutively and in order.
- **Train:** video order is shuffled every epoch; one random crop + horizontal flip per video, the same for all of its frames.
- **Eval:** centre crop, no flip (deterministic).
- Decoding runs on the CPU with **1 FFmpeg thread** per worker (OpenCV's default of 16 threads per video, multiplied by all the workers, used about 5x more RAM and overloaded the CPU). Everything after that (normalisation, the model, the loss, the optimizer) runs on the GPU.
- Workers return **uint8** tensors (4x less host-to-GPU traffic). Normalisation happens on the GPU.

In [4]:
def open_video(path):
    # 1 decode thread per video: the default (16) costs ~5x more RAM per open video and,
    # summed over all workers, oversubscribes the 24 cores.
    return cv2.VideoCapture(path, cv2.CAP_FFMPEG, [cv2.CAP_PROP_N_THREADS, 1])


def frame_count(path):
    cap = open_video(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


SEG_FRAMES = CLIP_LEN * CLIPS_PER_SEGMENT


def read_frames(path, start, count, y, x):
    """Decodes `count` consecutive frames from frame `start` (count=None: up to the end of the video).
       Every frame is resized to RESIZE_HW and cropped at (y, x). Returns a list of uint8 (CROP, CROP, 3) RGB arrays."""
    cap = open_video(path)
    if start > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    def grab():
        frames = []
        while count is None or len(frames) < count:
            ok, fr = cap.read()
            if not ok:
                break
            fr = cv2.resize(fr, (RESIZE_HW[1], RESIZE_HW[0]), interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(fr[y:y + CROP, x:x + CROP], cv2.COLOR_BGR2RGB))
        return frames

    frames = grab()
    if not frames and start > 0:            # seek failed -> reopen and step through the frames before `start`
        cap.release(); cap = open_video(path)
        for _ in range(start):
            if not cap.grab():
                break
        frames = grab()
    cap.release()
    return frames


class FullVideoDataset(Dataset):
    """One item = one segment of one video: up to CLIPS_PER_SEGMENT consecutive 16-frame clips, (N, C, T, H, W) uint8.
       The segments of a video together contain every one of its frames.
       Items are requested with keys (vid, seg, y, x, flip, pass) produced by VideoOrderSampler."""
    def __init__(self, items):
        self.items = items
        self.n_frames = [frame_count(p) for p, _ in items]
        self.n_segs = [max(1, -(-n // SEG_FRAMES)) for n in self.n_frames]

    def __len__(self):
        return sum(self.n_segs)

    def __getitem__(self, key):
        vid, seg, y, x, flip, pss = key
        path, label = self.items[vid]
        last = seg == self.n_segs[vid] - 1
        frames = read_frames(path, seg * SEG_FRAMES, None if last else SEG_FRAMES, y, x)   # last segment: read to the end
        if not frames and seg == 0:
            print(f"WARNING: could not decode {path}")
            frames = [np.zeros((CROP, CROP, 3), np.uint8)]
        if frames:
            frames += [frames[-1]] * (-len(frames) % CLIP_LEN)    # pad the last clip by repeating the last frame
        n_clips = len(frames) // CLIP_LEN
        clips = np.stack(frames) if frames else np.zeros((0, CROP, CROP, 3), np.uint8)
        if flip:
            clips = np.ascontiguousarray(clips[:, :, ::-1])
        clips = torch.from_numpy(clips).view(n_clips, CLIP_LEN, CROP, CROP, 3).permute(0, 4, 1, 2, 3)   # (N, C, T, H, W)
        return clips, label, vid, last, pss


class VideoOrderSampler(Sampler):
    """Yields segment keys video after video, so all segments of a video arrive together and in order
       (the DataLoader keeps the sampler's order even with many workers).
       train=True: video order shuffled; one random crop + flip per video, shared by all of its frames.
       passes=2 (stage 2): every video is streamed twice, see run_epoch."""
    def __init__(self, ds, train, passes=1):
        self.ds, self.train, self.passes = ds, train, passes

    def __len__(self):
        return sum(self.ds.n_segs) * self.passes

    def __iter__(self):
        order = list(range(len(self.ds.items)))
        if self.train:
            random.shuffle(order)
        for v in order:
            if self.train:
                y, x = random.randint(0, RESIZE_HW[0] - CROP), random.randint(0, RESIZE_HW[1] - CROP)
                flip = random.random() < 0.5
            else:
                y, x, flip = (RESIZE_HW[0] - CROP) // 2, (RESIZE_HW[1] - CROP) // 2, False
            for p in range(self.passes):
                for s in range(self.ds.n_segs[v]):
                    yield (v, s, y, x, flip, p)


def worker_init(_):
    cv2.setNumThreads(1)                     # one decode thread per worker; parallelism comes from workers


def make_loader(ds, sampler, persistent=False):
    """batch_size=None: every item is already a batch of clips.
       persistent=True keeps workers alive between epochs (train only); otherwise they exit when the loop ends."""
    return DataLoader(ds, sampler=sampler, batch_size=None,
                      num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init,
                      persistent_workers=persistent and NUM_WORKERS > 0,
                      prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None)


def close_loader(loader):
    """Stops persistent workers now. Without this, an interrupted cell keeps them alive (Jupyter holds the
       traceback), and every re-run starts another set on top until RAM runs out."""
    it = getattr(loader, "_iterator", None)
    if it is not None and hasattr(it, "_shutdown_workers"):
        it._shutdown_workers()
    loader._iterator = None


MEAN = torch.tensor([0.43216, 0.394666, 0.37645], device=device).view(1, 3, 1, 1, 1)   # Kinetics stats
STD  = torch.tensor([0.22803, 0.22145, 0.216989], device=device).view(1, 3, 1, 1, 1)

def to_input(x):
    x = x.to(device, non_blocking=True).float().div_(255)
    return ((x - MEAN) / STD).contiguous(memory_format=torch.channels_last_3d)


train_ds, val_ds, test_ds = FullVideoDataset(train_items), FullVideoDataset(val_items), FullVideoDataset(test_items)
for name, ds in [("TRAIN", train_ds), ("VAL", val_ds), ("TEST", test_ds)]:
    n = np.array(ds.n_frames)
    print(f"{name:5s}: {n.sum():8d} frames -> {(-(-n // CLIP_LEN)).sum():6d} clips | "
          f"frames/video min {n.min()} median {int(np.median(n))} max {n.max()}")

TRAIN:  2393787 frames -> 149747 clips | frames/video min 203 median 2236 max 628020
VAL  :   420393 frames ->  26299 clips | frames/video min 329 median 2714 max 126553
TEST :   361546 frames ->  22643 clips | frames/video min 209 median 1679 max 107994


## 5. Model: pretrained R(2+1)D-18 with a new binary head, applied to the whole video

In [5]:
def build_model():
    model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 1))   # 512 -> 1 logit
    return model.to(device).to(memory_format=torch.channels_last_3d)

n_pos = sum(l for _, l in train_items)
pos_weight = torch.tensor([(len(train_items) - n_pos) / max(n_pos, 1)], device=device)  # #normal / #anomaly in TRAIN
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print("pos_weight:", round(pos_weight.item(), 3))


def set_mode(model, phase):
    """Frozen parts stay in eval mode (BatchNorm uses pretrained statistics)."""
    model.eval()
    model.fc.train()
    if phase == "finetune":
        model.layer4.train()
        for m in model.layer4.modules():     # small batches -> keep BN running stats frozen
            if isinstance(m, nn.BatchNorm3d):
                m.eval()


def trunk(model, x):
    """stem + layer1-3 (frozen in both stages)."""
    return model.layer3(model.layer2(model.layer1(model.stem(x))))


def clip_features(model, h):
    """layer4 + global average pool -> one 512-d feature per clip."""
    return model.avgpool(model.layer4(h)).flatten(1)


def run_epoch(model, loader, optimizer=None, phase=None):
    """Runs every video of the loader through the model; returns one logit per video.
       Per video, the features of all of its clips are summed segment by segment; after the last segment their mean
       (the whole-video feature) goes through fc -> 1 logit.
       head:     the backbone is frozen, so only fc gets gradients.
       finetune: the video is streamed twice. Pass 0 (no grad) gives the video feature m = mean of the N clip
                 features, the loss and dL/dm. Pass 1 recomputes the clip features with gradients through layer4 and
                 backpropagates dL/dm / N into each of them. This is exactly the gradient of the whole-video loss, without
                 keeping the layer4 activations of thousands of clips in memory."""
    train = optimizer is not None
    if train:
        set_mode(model, phase)
        optimizer.zero_grad(set_to_none=True)
    else:
        model.eval()
    n_videos = len(loader.dataset.items)
    logits = np.zeros(n_videos, np.float32)
    feat_sum, n_clips, done = 0, 0, 0
    for clips, label, vid, last, pss in tqdm(loader, leave=False, desc=f"train ({phase})" if train else "eval"):
        x = to_input(clips) if len(clips) else None
        video_done = False
        if pss == 0:
            if x is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
                    feat_sum = feat_sum + clip_features(model, trunk(model, x)).float().sum(0)
                n_clips += len(clips)
            if last:
                feat = (feat_sum / n_clips).detach().requires_grad_(train and phase == "finetune")
                with torch.set_grad_enabled(train):
                    logit = model.fc(feat)                                    # whole-video feature -> 1 logit
                    if train:
                        loss = criterion(logit, torch.tensor([float(label)], device=device))
                        (loss / ACCUM_VIDEOS).backward()
                logits[vid] = logit.item()
                if train and phase == "finetune":
                    grad_clip = feat.grad / n_clips                           # dL/d(clip feature), the same for every clip
                else:
                    video_done = True
                feat_sum, n_clips = 0, 0
        else:
            if x is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
                    h = trunk(model, x)
                for c in h.split(FT_CHUNK):
                    with torch.autocast("cuda", dtype=AMP_DTYPE):
                        f = clip_features(model, c)
                    (f.float() @ grad_clip).sum().backward()
            video_done = last
        if train and video_done:
            done += 1
            if done % ACCUM_VIDEOS == 0 or done == n_videos:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
    return logits


def loss_acc(logits, items):
    y = np.array([l for _, l in items], np.float32)
    loss = criterion(torch.from_numpy(logits).to(device), torch.from_numpy(y).to(device)).item()
    return loss, ((logits > 0) == y).mean()


def atomic_save(obj, path):
    """Write to a temp file then rename, so an interrupted save never corrupts the checkpoint."""
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)


def get_rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    # RNG states must be CPU ByteTensors; torch.load(map_location=device) moved them to the GPU
    random.setstate(s["python"]); np.random.set_state(s["numpy"]); torch.set_rng_state(s["torch"].cpu())
    if s["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def fit(model, optimizer, scheduler, epochs, phase):
    """Trains `phase` for `epochs`, saving after every epoch to runs/:
         {phase}_last.pt -> model + optimizer + scheduler + history + RNG (used to resume)
         {phase}_best.pt -> best weights by val accuracy
       Re-running the cell resumes from the last finished epoch; a finished phase just reloads its best model."""
    last_path = os.path.join(RUNS_DIR, f"{phase}_last.pt")
    best_path = os.path.join(RUNS_DIR, f"{phase}_best.pt")
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc"]}
    best_acc, start_ep = -1.0, 1
    if RESUME and os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        set_rng_state(ck["rng"])
        hist, best_acc, start_ep = ck["history"], ck["best_acc"], ck["epoch"] + 1
        print(f"[{phase}] resumed from {last_path}: {ck['epoch']} epoch(s) done, best val acc {best_acc:.4f}")

    if start_ep <= epochs:
        train_loader = make_loader(train_ds, VideoOrderSampler(train_ds, train=True,
                                                               passes=2 if phase == "finetune" else 1), persistent=True)
        val_loader = make_loader(val_ds, VideoOrderSampler(val_ds, train=False))
    try:
        for ep in range(start_ep, epochs + 1):
            t0 = time.time()
            tr_loss, tr_acc = loss_acc(run_epoch(model, train_loader, optimizer, phase), train_items)
            va_loss, va_acc = loss_acc(run_epoch(model, val_loader), val_items)
            scheduler.step()
            for k, v in zip(hist, [tr_loss, tr_acc, va_loss, va_acc]):
                hist[k].append(v)
            flag = ""
            if va_acc > best_acc:
                best_acc = va_acc
                atomic_save({"model": model.state_dict(), "epoch": ep, "val_acc": va_acc, "phase": phase}, best_path)
                flag = "  <- best, saved"
            atomic_save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                         "scheduler": scheduler.state_dict(), "rng": get_rng_state(),
                         "history": hist, "best_acc": best_acc, "epoch": ep}, last_path)
            print(f"[{phase}] epoch {ep:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.4f} | {time.time()-t0:.0f}s | {mem_status()}{flag}")
    finally:
        if start_ep <= epochs:
            close_loader(train_loader)
    best = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best["model"])
    print(f"Loaded best {phase} model from {best_path} (epoch {best['epoch']}, val acc {best['val_acc']:.4f})")
    return hist

pos_weight: 0.8


## 6. Plotting and metric helpers

In [6]:
def plot_history(hist, title, ft_start=None, save_name=None):
    ep = np.arange(1, len(hist["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, key, name in [(axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy")]:
        ax.plot(ep, hist[f"train_{key}"], "o-", label=f"Train {name.lower()}")
        ax.plot(ep, hist[f"val_{key}"], "s-", label=f"Val {name.lower()}")
        if ft_start is not None:
            ax.axvline(ft_start, color="red", ls="--", lw=2, label="Fine-tuning started (layer4 unfrozen)")
        ax.set_xlabel("Epoch"); ax.set_ylabel(name); ax.set_title(f"{title} — {name}")
        ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(RUNS_DIR, save_name), dpi=150)
    plt.show()


@torch.no_grad()
def predict(model, ds):
    """Anomaly probability of every video, each computed from all of its frames (centre crop)."""
    logits = run_epoch(model, make_loader(ds, VideoOrderSampler(ds, train=False)))
    return 1 / (1 + np.exp(-logits.astype(np.float64))), np.array([l for _, l in ds.items])


def report(model, ds, split_name, stage_name):
    items = ds.items
    if not items:
        print(f"{split_name}: no labelled videos, skipped"); return None
    probs, y = predict(model, ds)
    pred = (probs >= 0.5).astype(int)
    m = {"accuracy": accuracy_score(y, pred),
         "precision": precision_score(y, pred, zero_division=0),
         "recall": recall_score(y, pred, zero_division=0),
         "f1": f1_score(y, pred, zero_division=0),
         "roc_auc": roc_auc_score(y, probs) if len(set(y)) == 2 else float("nan")}
    print(f"===== {stage_name} | {split_name} ({len(items)} videos, every frame) =====")
    print(pd.DataFrame([m], index=[split_name]).round(4).to_string())
    print(classification_report(y, pred, target_names=CLASSES, digits=4, zero_division=0))

    df = pd.DataFrame({"video": [os.path.relpath(p, os.path.join(DATASET_ROOT, "Videos")) for p, _ in items],
                       "category": [category(p) for p, _ in items],
                       "label": np.array(CLASSES)[y], "anomaly_prob": probs.round(4),
                       "prediction": np.array(CLASSES)[pred]})
    df["correct"] = df["label"] == df["prediction"]
    print("Per-category accuracy:")
    print(df.groupby("category")["correct"].agg(["mean", "size"]).rename(columns={"mean": "acc", "size": "videos"})
            .round(3).to_string(), "\n")

    cm = confusion_matrix(y, pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{stage_name} — {split_name}")
    plt.tight_layout(); plt.show()
    return m


def predict_unlabelled(model, ds, stage_name):
    """No labels, no metrics: saves the anomaly probability and predicted class of every video to a CSV."""
    items = ds.items
    if not items:
        print("TEST: no videos, skipped"); return None
    probs, _ = predict(model, ds)
    df = pd.DataFrame({"video": [os.path.relpath(p, os.path.join(DATASET_ROOT, "Videos")) for p, _ in items],
                       "anomaly_prob": probs.round(4),
                       "prediction": np.where(probs >= 0.5, CLASSES[1], CLASSES[0])})
    out = os.path.join(RUNS_DIR, f"test_predictions_{stage_name.split()[0].lower()}.csv")
    df.to_csv(out, index=False)
    print(f"===== {stage_name} | TEST ({len(df)} videos) -> {out} =====")
    print(df["prediction"].value_counts().to_string())
    return df

## 7. Stage 1 — Feature extraction (frozen backbone, train only the head)

In [ ]:
model = build_model()
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

opt = torch.optim.AdamW(model.fc.parameters(), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HEAD_EPOCHS)
hist_head = fit(model, opt, sched, HEAD_EPOCHS, "head")

trainable params: 513


train (head):   2%|▏         | 77/4817 [00:06<04:28, 17.62it/s] 

### Stage 1 — accuracy & loss curves

In [ ]:
plot_history(hist_head, "Stage 1 (feature extraction)", save_name="curves_stage1.png")

### Stage 1 — accuracy, precision, recall, F1, ROC-AUC (VAL) + TEST predictions

In [ ]:
metrics = {}
metrics[("Feature extraction", "VAL")]  = report(model, val_ds, "VAL", "Feature extraction")
test_pred_head = predict_unlabelled(model, test_ds, "Feature extraction")

## 8. Stage 2 — Fine-tuning (unfreeze `layer4`, the last of the 4 residual stages)

Starts from the best Stage 1 checkpoint (already loaded in `model`). Each training video is streamed twice per epoch (pass 0: whole-video feature + loss, pass 1: gradients through `layer4` for every clip), see `run_epoch`.

In [ ]:
for p in model.layer4.parameters():
    p.requires_grad = True
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

opt = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": LR_FT_L4},
    {"params": model.fc.parameters(),     "lr": LR_FT_HEAD},
], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
torch.cuda.empty_cache()
hist_ft = fit(model, opt, sched, FT_EPOCHS, "finetune")
torch.save(model.state_dict(), os.path.join(RUNS_DIR, "final_model_weights.pt"))   # plain state_dict of the best fine-tuned model

### Stage 1 + Stage 2 — accuracy & loss curves (red line = fine-tuning started)

In [ ]:
hist_all = {k: hist_head[k] + hist_ft[k] for k in hist_head}
plot_history(hist_all, "Feature extraction → Fine-tuning", ft_start=len(hist_head["train_loss"]),
             save_name="curves_stage1_stage2.png")

### Stage 2 — accuracy, precision, recall, F1, ROC-AUC (VAL) + TEST predictions

In [ ]:
metrics[("Fine-tuned (layer4)", "VAL")]  = report(model, val_ds, "VAL", "Fine-tuned (layer4)")
test_pred_ft = predict_unlabelled(model, test_ds, "Fine-tuned (layer4)")

## 9. Comparison: feature extraction vs fine-tuning

In [ ]:
summary = pd.DataFrame({k: v for k, v in metrics.items() if v is not None}).T
summary.index.names = ["stage", "split"]
summary.to_csv(os.path.join(RUNS_DIR, "metrics_summary.csv"))
summary.round(4)